In [1]:
from datasets import load_dataset

ds = load_dataset("Sabrina1763/wikipedia_french")

/home/koubra/Documents/Projet_IA/Chadian arabic update/.env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating test split: 100%|██████████| 50000/50000 [00:02<00:00, 21527.79 examples/s]


In [2]:
ds

DatasetDict({
    train: Dataset({
        features: ['title', 'text'],
        num_rows: 400000
    })
    validation: Dataset({
        features: ['title', 'text'],
        num_rows: 50000
    })
    test: Dataset({
        features: ['title', 'text'],
        num_rows: 50000
    })
})

In [5]:
import pandas as pd
import csv

# Source - https://stackoverflow.com/a/8685873
# Posted by Ricardo Cárdenes, modified by community. See post 'Timeline' for change history
# Retrieved 2026-10-17, License - CC BY-SA 4.0

with open('dict.csv', 'w') as csv_file:  
    writer = csv.writer(csv_file)
    for key, value in ds.items():
       writer.writerow([key, value])


In [1]:
from datasets import load_dataset
ds = load_dataset("LeFrenchNewsLab/frenchnews-7", split="train")
test = ds.filter(lambda r: r["split"] == "test")


/home/koubra/Documents/Projet_IA/Chadian arabic update/.env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Filter: 100%|██████████| 87637/87637 [00:00<00:00, 159643.26 examples/s]


In [8]:
len(ds)

87637

In [1]:
import requests
from bs4 import BeautifulSoup
from datasets import load_dataset
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# 1. Chargement du manifest officiel (contient les URLs, dates, éditeurs et catégories)
print("Téléchargement du manifest FrenchNews-7 depuis Hugging Face...")
dataset = load_dataset("LeFrenchNewsLab/frenchnews-7")

# Conversion en DataFrame Pandas pour faciliter la manipulation (split 'train')
df = pd.DataFrame(dataset['train'])

# Optionnel : Filtrer uniquement sur la catégorie "Société" (qui regroupe les faits divers)
# df_faits_divers = df[df['label'] == 'Société']

print(f"Nombre total d'URLs à traiter : {len(df)}")

# 2. Fonction d'extraction du contenu textuel de l'article à partir de son URL
def fetch_article_text(url):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # Extraction basique des paragraphes de texte
            # (Note : Selon l'éditeur, vous devrez parfois cibler une balise spécifique comme <article>)
            paragraphs = soup.find_all('p')
            article_text = " ".join([p.get_text() for p in paragraphs]).strip()
            
            return article_text if len(article_text) > 100 else None
    except Exception as e:
        return None
    return None

# 3. Fonction pour exécuter l'extraction en parallèle (Multi-threading pour aller plus vite)
def reconstruct_dataset(dataframe, max_workers=10):
    urls = dataframe['url'].tolist()
    texts = []
    
    print(f"Extraction des articles en cours (avec {max_workers} workers)...")
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Utilisation de tqdm pour afficher une barre de progression
        results = list(tqdm(executor.map(fetch_article_text, urls), total=len(urls)))
    
    dataframe['text'] = results
    # On supprime les lignes où l'extraction a échoué (ex: paywall, URL morte)
    dataframe_clean = dataframe.dropna(subset=['text'])
    return dataframe_clean

# --- Exemple d'exécution sur un échantillon (ex: les 100 premiers articles) ---
# Enlevez le 'head(100)' pour lancer la reconstruction sur l'intégralité du dataset (87 637 lignes)
df_sample = df.head(100).copy()
df_final = reconstruct_dataset(df_sample, max_workers=10)

# 4. Sauvegarde de votre dataset reconstitué, prêt pour la backtranslation
df_final.to_csv("frenchnews7_reconstituted.csv", index=False, encoding="utf-8")
print(f"Extraction terminée ! {len(df_final)} articles sauvegardés sous 'frenchnews7_reconstituted.csv'.")


/home/koubra/Documents/Projet_IA/Chadian arabic update/.env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Téléchargement du manifest FrenchNews-7 depuis Hugging Face...
Nombre total d'URLs à traiter : 87637
Extraction des articles en cours (avec 10 workers)...


100%|██████████| 100/100 [00:09<00:00, 10.34it/s]

Extraction terminée ! 0 articles sauvegardés sous 'frenchnews7_reconstituted.csv'.


In [10]:
# Source - https://stackoverflow.com/q/67998191
# Posted by Cranjis, modified by community. See post 'Timeline' for change history
# Retrieved 2026-10-20, License - CC BY-SA 4.0

!jupyter nbextension enable --py widgetsnbextension

usage: jupyter [-h] [--version] [--config-dir] [--data-dir] [--runtime-dir]
               [--paths] [--json] [--debug]
               [subcommand]

Jupyter: Interactive Computing

positional arguments:
  subcommand     the subcommand to launch

options:
  -h, --help     show this help message and exit
  --version      show the versions of core jupyter packages and exit
  --config-dir   show Jupyter config dir
  --data-dir     show Jupyter data dir
  --runtime-dir  show Jupyter runtime dir
  --paths        show all Jupyter paths. Add --json for machine-readable
                 format.
  --json         output paths as machine-readable json
  --debug        output debug information about paths

Available subcommands: dejavu events execute kernel kernelspec migrate
nbconvert run server troubleshoot trust

Jupyter command `jupyter-nbextension` not found.


In [11]:
import pandas as pd
from datasets import load_dataset
import trafilatura
from bs4 import BeautifulSoup
import cloudscraper
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# 1. Initialisation du scraper anti-bot et chargement du manifest FrenchNews-7
scraper = cloudscraper.create_scraper()
print("Téléchargement du manifest FrenchNews-7 depuis Hugging Face...")
dataset = load_dataset("LeFrenchNewsLab/frenchnews-7")
df = pd.DataFrame(dataset['train'])

def extract_clean_press_text(url):
    try:
        # Requête robuste pour contourner Cloudflare
        response = scraper.get(url, timeout=10)
        if response.status_code != 200:
            return None
            
        html_content = response.text
        soup = BeautifulSoup(html_content, 'html.parser')
        
        # --- LOGIQUE SPÉCIFIQUE PAR ÉDITEUR (Règles de filtrage du projet) ---
        if "lemonde.fr" in url:
            # Le Monde protège son contenu, on cible la zone article principale
            article_body = soup.find(class_="article__content") or soup.find(id="article-body")
            if article_body:
                html_content = str(article_body)
                
        elif "lefigaro.fr" in url:
            # Le Figaro utilise la classe 'fig-paragraph' ou le conteneur 'fig-content'
            paragraphs = soup.find_all(class_="fig-paragraph")
            if paragraphs:
                html_content = " ".join([p.get_text() for p in paragraphs])
            else:
                article_body = soup.find(class_="fig-content")
                if article_body: html_content = str(article_body)
                
        elif "liberation.fr" in url:
            # Libération structure ses textes dans des sections dédiées
            article_body = soup.find(class_="article-body") or soup.find(class_="read-left-wrapper")
            if article_body:
                html_content = str(article_body)
                
        elif "francetvinfo.fr" in url:
            # FranceInfo isole le texte dans le corps 'text'
            article_body = soup.find(class_="text") or soup.find(id="article")
            if article_body:
                html_content = str(article_body)

        # --- EXTRACTEUR DE TEXTE AVANCÉ (Fallback Trafilatura) ---
        # Trafilatura nettoie automatiquement les cookies, publicités et menus restants
        clean_text = trafilatura.extract(html_content, 
                                         include_comments=False, 
                                         include_tables=False,
                                         no_fallback=False)
        
        # Validation réglementaire du volume de texte (seuil de l'équipe : ~200 caractères minimum)
        if clean_text and len(clean_text) > 200:
            return clean_text.strip()
            
    except Exception:
        pass
    return None

# 3. Pipeline d'exécution parallélisé
def build_frenchnews7_dataset(dataframe, max_workers=5):
    urls = dataframe['url'].tolist()
    
    print(f"Reconstitution optimisée en cours pour {len(urls)} articles...")
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(tqdm(executor.map(extract_clean_press_text, urls), total=len(urls)))
    
    dataframe['text'] = results
    # On élimine les articles sous paywall strict ou liens morts
    clean_df = dataframe.dropna(subset=['text'])
    return clean_df

# --- TEST SUR UN ÉCHANTILLON ---
# Conseil pour vos Faits Divers : Utilisez 
df[df['label'] == 'Société']
df_sample = df.head(10000).copy() 
df_final = build_frenchnews7_dataset(df_sample, max_workers=5)
df_end = df_final["text"]

# Sauvegarde finale
#df_end.to_csv("frenchnews7_reconstituted.csv", index=False, encoding="utf-8")


Téléchargement du manifest FrenchNews-7 depuis Hugging Face...
Reconstitution optimisée en cours pour 10000 articles...


100%|██████████| 10000/10000 [56:40<00:00,  2.94it/s] 


In [12]:
with open("molingal_data_french.txt", "w") as file :
    file.writelines(df_end)
print(f"Succès ! {len(df_end)} articles parfaitement nettoyés et extraits.")


Succès ! 9456 articles parfaitement nettoyés et extraits.


In [13]:
df_end.head(10)

0    Fin mai, Hadrien Canter, fondateur de la start...
1    La revue des revues. Depuis plusieurs années, ...
2    Le chargeur solaire USB est l’ami des batterie...
3    Les projets spatiaux du multimilliardaire Elon...
4    C’est l’histoire d’une maladie infectieuse don...
5    La nouvelle de la suppression de données de la...
6    D’un côté, on fustige les lourdeurs liées à la...
7    Notre société compte sur eux pour imaginer les...
8    Les plantes carnivores fascinent, par l’origin...
9    Le bien-être d’un cheval monté en compétition ...
Name: text, dtype: str